In [6]:
!pip install psycopg2-binary deepface retina-face ultralytics networkx geopandas shapely folium opencv-python numpy

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ----------------------------- ---------- 1.0/1.4 MB 10.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.0/1.4 MB 10.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.0/1.4 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 9.7 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 10.4 MB/s  0:00:00
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   -------------- ----

In [12]:
import psycopg2
from psycopg2.extras import execute_values
import numpy as np

# Connect to local PostgreSQL
conn = psycopg2.connect(
    dbname="missingperson",
    user="postgres",
    password="101207",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# Enable pgvector extension
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Create missing persons registry table
cursor.execute("""
CREATE TABLE IF NOT EXISTS missing_persons (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    last_worn_clothes TEXT,
    last_seen_location VARCHAR(255),
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    photo_path TEXT,
    face_embedding vector(512) -- ArcFace generates 512-dimensional embeddings
);
""")

# Create sighting logs table
cursor.execute("""
CREATE TABLE IF NOT EXISTS sightings (
    sighting_id SERIAL PRIMARY KEY,
    person_id INT REFERENCES missing_persons(id),
    sighted_lat DOUBLE PRECISION,
    sighted_lon DOUBLE PRECISION,
    camera_id VARCHAR(50),
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    similarity_score FLOAT
);
""")
conn.commit()
print("PostgreSQL & pgvector schema initialized successfully.")

PostgreSQL & pgvector schema initialized successfully.


In [14]:
from deepface import DeepFace

def register_missing_person(name, clothes, location, lat, lon, image_path):
    # Generate 512-d embedding using ArcFace
    representation = DeepFace.represent(
        img_path=image_path,
        model_name="ArcFace",
        detector_backend="retinaface",
        enforce_detection=True
    )
    embedding = representation[0]["embedding"]

    insert_query = """
    INSERT INTO missing_persons (name, last_worn_clothes, last_seen_location, latitude, longitude, photo_path, face_embedding)
    VALUES (%s, %s, %s, %s, %s, %s, %s) RETURNING id;
    """
    cursor.execute(insert_query, (name, clothes, location, lat, lon, image_path, embedding))
    person_id = cursor.fetchone()[0]
    conn.commit()
    print(f"Registered {name} successfully with ID: {person_id}")
    return person_id

# Example registration:
# register_missing_person("John Doe", "Red hoodie, blue jeans", "Central Market", 15.8281, 78.0373, "missing_person.jpg")

In [15]:
import cv2
from ultralytics import YOLO

# Load YOLOv8 for person detection in crowded footage
yolo_model = YOLO("yolov8n.pt")

def query_best_match(face_crop):
    try:
        # Extract embedding from detected face
        rep = DeepFace.represent(
            img_path=face_crop,
            model_name="ArcFace",
            detector_backend="skip", # bounding box already identified
            enforce_detection=False
        )
        if not rep:
            return None
        emb = rep[0]["embedding"]
        
        # pgvector cosine distance search (<=> operator)
        search_query = """
        SELECT id, name, last_worn_clothes, (face_embedding <=> %s::vector) AS distance
        FROM missing_persons
        ORDER BY distance ASC
        LIMIT 1;
        """
        cursor.execute(search_query, (emb,))
        match = cursor.fetchone()
        
        # ArcFace cosine distance threshold ~0.68 (lower = closer match)
        if match and match[3] < 0.60:
            return {"id": match[0], "name": match[1], "clothes": match[2], "distance": match[3]}
    except Exception as e:
        pass
    return None

def process_stream(source=0, camera_lat=15.8285, camera_lon=78.0380):
    """
    source: 0 for live webcam, or "path/to/video.mp4" for stored CCTV video
    """
    cap = cv2.VideoCapture(source)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Object detection: identify people (class 0 is person in COCO)
        results = yolo_model(frame, classes=[0], verbose=False)
        
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                person_crop = frame[y1:y2, x1:x2]
                
                if person_crop.size == 0:
                    continue
                
                # Check match against database
                match = query_best_match(person_crop)
                if match:
                    # Visual Alert on Frame
                    label = f"MATCH: {match['name']} ({1 - match['distance']:.2f})"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    
                    # Log sighting
                    cursor.execute("""
                    INSERT INTO sightings (person_id, sighted_lat, sighted_lon, camera_id, similarity_score)
                    VALUES (%s, %s, %s, %s, %s);
                    """, (match['id'], camera_lat, camera_lon, "CAM_01", float(1 - match['distance'])))
                    conn.commit()
                    print(f"ALERT: {match['name']} detected at [{camera_lat}, {camera_lon}]!")
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)

        cv2.imshow("Crowd Search & Video Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run on webcam: process_stream(source=0)
# Run on stored video: process_stream(source="surveillance_footage.mp4")

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Syam Sundar\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [16]:
import networkx as nx

def build_city_network():
    # Directed Graph modeling local road network
    G = nx.DiGraph()
    
    # Intersections / checkpoints (lat, lon)
    nodes = {
        "Sighting_Point": (15.8285, 78.0380),
        "Junction_A": (15.8300, 78.0395),
        "Junction_B": (15.8260, 78.0350),
        "NGO_Station_Alpha": (15.8350, 78.0420),
        "NGO_Station_Beta": (15.8220, 78.0310)
    }
    for node, coords in nodes.items():
        G.add_node(node, pos=coords)
        
    # Directed weighted streets (weight = travel time / distance in minutes)
    edges = [
        ("NGO_Station_Alpha", "Junction_A", 4),
        ("Junction_A", "Sighting_Point", 3),
        ("Sighting_Point", "Junction_B", 2),
        ("NGO_Station_Beta", "Junction_B", 5),
        ("Junction_B", "Sighting_Point", 3),
        ("Sighting_Point", "Junction_A", 3),
    ]
    G.add_weighted_edges_from(edges)
    return G

def find_fastest_ngo_route(G, sighting_node="Sighting_Point"):
    ngo_stations = ["NGO_Station_Alpha", "NGO_Station_Beta"]
    best_route = None
    min_dist = float("inf")
    closest_ngo = None

    for ngo in ngo_stations:
        try:
            length = nx.shortest_path_length(G, source=ngo, target=sighting_node, weight="weight")
            if length < min_dist:
                min_dist = length
                best_route = nx.shortest_path(G, source=ngo, target=sighting_node, weight="weight")
                closest_ngo = ngo
        except nx.NetworkXNoPath:
            continue
            
    return closest_ngo, min_dist, best_route

city_graph = build_city_network()
ngo, eta, path = find_fastest_ngo_route(city_graph)
print(f"Nearest Responder: {ngo} | Estimated Travel Time: {eta} mins | Optimal Route: {' -> '.join(path)}")

Nearest Responder: NGO_Station_Alpha | Estimated Travel Time: 7 mins | Optimal Route: NGO_Station_Alpha -> Junction_A -> Sighting_Point


In [17]:
import geopandas as gpd
from shapely.geometry import Point, LineString
import folium

def generate_geo_alert(G, path, containment_radius_meters=300):
    route_coords = [G.nodes[node]["pos"] for node in path]
    sighting_coord = route_coords[-1]
    
    # 1. Containment Perimeter (buffer in approximate degrees)
    buffer_deg = containment_radius_meters / 111320.0
    perimeter = Point(sighting_coord[1], sighting_coord[0]).buffer(buffer_deg)
    
    # 2. Convert to GeoDataFrame
    route_geom = LineString([(lon, lat) for lat, lon in route_coords])
    gdf_route = gpd.GeoDataFrame([{"geometry": route_geom, "name": "Rescue Route"}], crs="EPSG:4326")
    gdf_perimeter = gpd.GeoDataFrame([{"geometry": perimeter, "name": "Containment Grid"}], crs="EPSG:4326")
    
    # Export standard GeoJSON for NGOs / QGIS
    gdf_route.to_file("ngo_route.geojson", driver="GeoJSON")
    gdf_perimeter.to_file("containment_grid.geojson", driver="GeoJSON")
    
    # 3. Interactive Leaflet Map for field units
    m = folium.Map(location=sighting_coord, zoom_start=15)
    folium.Marker(route_coords[0], popup=f"Starting: {path[0]}", icon=folium.Icon(color="blue", icon="shield")).add_to(m)
    folium.Marker(sighting_coord, popup="Missing Person Sighted", icon=folium.Icon(color="red", icon="warning")).add_to(m)
    folium.PolyLine(route_coords, color="red", weight=4, opacity=0.8, tooltip="Fastest Route").add_to(m)
    folium.Circle(location=sighting_coord, radius=containment_radius_meters, color="orange", fill=True, fill_opacity=0.2).add_to(m)
    
    m.save("rescue_dispatch_map.html")
    print("Files 'ngo_route.geojson' and 'rescue_dispatch_map.html' exported successfully.")
    return m

alert_map = generate_geo_alert(city_graph, path)

Files 'ngo_route.geojson' and 'rescue_dispatch_map.html' exported successfully.


In [18]:
from IPython.display import HTML

dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', sans-serif; }
  body { background: #0f172a; color: #f8fafc; padding: 20px; }
  .grid-container { display: grid; grid-template-columns: 320px 1fr 340px; gap: 18px; height: 85vh; }
  .card { background: #1e293b; border: 1px solid #334155; border-radius: 10px; padding: 18px; overflow-y: auto; }
  h2 { font-size: 1.1rem; color: #38bdf8; margin-bottom: 14px; text-transform: uppercase; letter-spacing: 0.05em; }
  .field { margin-bottom: 12px; }
  label { display: block; font-size: 0.8rem; color: #94a3b8; margin-bottom: 4px; }
  input, select, textarea { width: 100%; padding: 8px 10px; background: #0f172a; border: 1px solid #475569; border-radius: 6px; color: #f8fafc; }
  button { width: 100%; padding: 10px; background: #2563eb; color: #fff; border: none; border-radius: 6px; font-weight: 600; cursor: pointer; transition: 0.2s; }
  button:hover { background: #1d4ed8; }
  .alert-banner { background: #ef444420; border-left: 4px solid #ef4444; padding: 12px; border-radius: 4px; margin-bottom: 12px; }
  .badge { display: inline-block; padding: 2px 8px; border-radius: 9999px; font-size: 0.75rem; font-weight: 600; }
  .badge-red { background: #dc2626; color: white; }
  .badge-green { background: #16a34a; color: white; }
  .feed-frame { width: 100%; height: 55%; background: #000; border-radius: 8px; display: flex; align-items: center; justify-content: center; color: #64748b; border: 1px dashed #475569; }
  .map-frame { width: 100%; height: 40%; margin-top: 15px; border-radius: 8px; border: 1px solid #334155; }
</style>
</head>
<body>

<div class="grid-container">
  <!-- Missing Person Registry Form -->
  <div class="card">
    <h2>Register Case</h2>
    <div class="field">
      <label>Full Name</label>
      <input type="text" placeholder="e.g. Syam Sundar" />
    </div>
    <div class="field">
      <label>Last Worn Clothing</label>
      <input type="text" placeholder="Navy pinstripe shirt, trousers" />
    </div>
    <div class="field">
      <label>Last Known Coordinates</label>
      <input type="text" placeholder="15.8285, 78.0380" />
    </div>
    <div class="field">
      <label>Face Reference Image</label>
      <input type="file" />
    </div>
    <button onclick="alert('Case synced with pgvector repository.')">Save to pgvector</button>
  </div>

  <!-- Center Screen: Live Detection Feed & Map -->
  <div class="card" style="display: flex; flex-direction: column;">
    <h2>Surveillance & Detection Engine</h2>
    <div class="feed-frame">
      <span>[Live Feed / Stored Video Stream - YOLO + ArcFace Inference Active]</span>
    </div>
    <iframe class="map-frame" src="rescue_dispatch_map.html"></iframe>
  </div>

  <!-- Real-time Alerts & NGO Dispatch -->
  <div class="card">
    <h2>Dispatch & Notifications</h2>
    <div class="alert-banner">
      <span class="badge badge-red">SIGHTING DETECTED</span>
      <p style="font-size: 0.85rem; margin-top: 6px; font-weight: bold;">Camera ID: CAM_01 (Crowded Plaza)</p>
      <p style="font-size: 0.8rem; color: #cbd5e1;">Confidence: 94.8% (ArcFace)</p>
    </div>

    <div style="margin-top: 15px;">
      <h3 style="font-size: 0.9rem; color: #94a3b8; margin-bottom: 8px;">Automated Shortest Path</h3>
      <p style="font-size: 0.8rem;">Assigned Unit: <strong>NGO_Station_Alpha</strong></p>
      <p style="font-size: 0.8rem;">ETA: <strong>7 mins</strong></p>
      <p style="font-size: 0.8rem;">Route: <code>Station Alpha -> Junction A -> Plaza</code></p>
    </div>

    <button style="margin-top: 20px; background: #059669;" onclick="alert('Dispatched GeoJSON coordinate package to nearest NGO field teams.')">
      Broadcast GeoJSON to NGO
    </button>
  </div>
</div>

</body>
</html>
"""

HTML(dashboard_html)